# Pipeline v2 ChatGPT + Voiceover Workbench

Resume-first notebook for the manual production path after App 4.

Flow:

1. Load an existing `video_packaging_result.json` by `RESUME_RUN_ID`.
2. Export a ChatGPT `content_packet` handoff.
3. Paste the handoff into ChatGPT and save the returned narration script as text.
4. Run `src.pipeline_v2.voiceover` from that manual script text.
5. Plan visuals from the script and voiceover, or export a ChatGPT visual-planning handoff.
6. Resolve assets, render scene clips, and assemble the final video.


In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_V2 = PROJECT_ROOT / "output_v2"
PIPELINE_V2_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline_v2" / "config" / "workbench.yaml"
NEWS_COLLECTION_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "apps" / "news_collection"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root:  {OUTPUT_V2}")
print(f"Config:       {PIPELINE_V2_CONFIG_PATH}")


Project root: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation
Output root:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2
Config:       /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/src/pipeline_v2/config/workbench.yaml


In [2]:
# Set this to a run that has completed App 4: Video Packaging.
# RESUME_RUN_ID = "v2_apps_2026_06_09_225504_daily_48h_all_sources"
# RESUME_RUN_ID = "v2_apps_2026_06_12_225334_daily_48h_all_sources"
RESUME_RUN_ID = "v2_apps_2026_06_26_231604_daily_48h_all_sources"

# Paste the full sectioned ChatGPT response here. The notebook will parse the script block.
CHATGPT_RESPONSE_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_response.txt"
MANUAL_SCRIPT_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_full_script.txt"
PARSED_RESPONSE_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_response_parsed.json"

WORKBENCH_CONFIG = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(WORKBENCH_CONFIG, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")

RUN_PROFILE = WORKBENCH_CONFIG.get("default_run_profile", "daily_48h_all_sources")
PROFILE_CONFIG = deepcopy(WORKBENCH_CONFIG["profiles"][RUN_PROFILE])

HANDOFF_OUTPUT_DIR = OUTPUT_V2 / "handoffs" / RESUME_RUN_ID
VOICEOVER_INPUT_DIR = OUTPUT_V2 / "voiceover_inputs"

print(f"Resume run:          {RESUME_RUN_ID}")
print(f"ChatGPT response:    {CHATGPT_RESPONSE_PATH}")
print(f"Parsed script path:  {MANUAL_SCRIPT_PATH}")
print(f"ChatGPT handoff dir: {HANDOFF_OUTPUT_DIR / 'chatgpt' / 'content_packet'}")


Resume run:          v2_apps_2026_06_26_231604_daily_48h_all_sources
ChatGPT response:    /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_scripts/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt_response.txt
Parsed script path:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_scripts/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt_full_script.txt
ChatGPT handoff dir: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/content_packet


In [3]:
def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return data


def load_optional_json(path: Path) -> dict:
    return load_json(path) if path.exists() else {}


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_V2": str(OUTPUT_V2),
        "RUN_ID": RESUME_RUN_ID,
    }
    if isinstance(value, str):
        return value.format(**context)
    if isinstance(value, list):
        return [render_config_templates(item) for item in value]
    if isinstance(value, dict):
        return {key: render_config_templates(item) for key, item in value.items()}
    return value


def voiceover_variant(config: dict) -> str:
    def slug(value: object) -> str:
        text = str(value).strip().lower().replace(".", "p")
        text = "".join(ch if ch.isalnum() else "_" for ch in text)
        return "_".join(part for part in text.split("_") if part)

    speed = config.get("speed")
    speed_label = f"speed_{slug(speed)}" if speed is not None else "style_speed"
    voice_label = slug(config.get("voice") or "default_voice")
    return "_".join(
        part for part in [slug(config.get("provider", "provider")), slug(config.get("style", "style")), voice_label, speed_label] if part
    )


def build_post_app4_manifest() -> dict:
    news_input = load_optional_json(OUTPUT_V2 / "news_collection" / RESUME_RUN_ID / "news_collection_input.json")
    evidence_input = load_optional_json(OUTPUT_V2 / "evidence_prep" / RESUME_RUN_ID / "evidence_prep_input.json")
    story_input = load_optional_json(OUTPUT_V2 / "story_mapping" / RESUME_RUN_ID / "story_mapping_input.json")
    video_input = load_optional_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_input.json")
    return {
        "run_id": RESUME_RUN_ID,
        "generated_at": datetime.now(IST).isoformat(),
        "pipeline_track": "pipeline_v2",
        "orchestrator": "notebooks/v2_chatgpt_voiceover_workbench.ipynb",
        "config_source": str(PIPELINE_V2_CONFIG_PATH.relative_to(PROJECT_ROOT)),
        "apps": {
            "news_collection": {"config": news_input.get("config", {})},
            "evidence_prep": {"config": evidence_input.get("config", {})},
            "story_mapping": {"config": story_input.get("config", {})},
            "video_packaging": {"config": video_input.get("config", {})},
        },
    }


def build_summary_bundle() -> dict:
    return {
        "news_collection": load_optional_json(OUTPUT_V2 / "news_collection" / RESUME_RUN_ID / "summary.json"),
        "evidence_prep": load_optional_json(OUTPUT_V2 / "evidence_prep" / RESUME_RUN_ID / "summary.json"),
        "story_mapping": load_optional_json(OUTPUT_V2 / "story_mapping" / RESUME_RUN_ID / "summary.json"),
        "video_packaging": load_optional_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "summary.json"),
    }

SECTION_MARKERS = ("NOTES", "SCRIPT", "METADATA")
PLACEHOLDER_RESPONSE_TEXT = "Paste the full CHATGPT_PIPELINE_RESPONSE_V1 response here, then rerun the parser cell.\n"


def parse_chatgpt_pipeline_response(text: str) -> dict:
    raw = text.strip()
    if not raw or raw.startswith("Paste the full CHATGPT_PIPELINE_RESPONSE_V1 response here"):
        raise ValueError("ChatGPT response file is empty or still a placeholder")
    if "CHATGPT_PIPELINE_RESPONSE_V1" not in raw[:500]:
        raise ValueError("ChatGPT response must start with or near CHATGPT_PIPELINE_RESPONSE_V1")

    marker_positions: list[tuple[int, str, int]] = []
    for marker in SECTION_MARKERS:
        token = f"[{marker}]"
        index = raw.find(token)
        if index == -1:
            raise ValueError(f"ChatGPT response is missing required section {token}")
        marker_positions.append((index, marker.lower(), index + len(token)))
    marker_positions.sort()

    sections = {}
    for position, name, content_start in marker_positions:
        next_positions = [item[0] for item in marker_positions if item[0] > position]
        content_end = min(next_positions) if next_positions else len(raw)
        sections[name] = raw[content_start:content_end].strip()

    script = clean_voiceover_script(sections["script"])
    if not script:
        raise ValueError("Parsed [SCRIPT] section is empty")
    warnings = validate_voiceover_script_text(script)
    return {
        "response_version": "CHATGPT_PIPELINE_RESPONSE_V1",
        "notes": sections.get("notes", ""),
        "script": script,
        "metadata_text": sections.get("metadata", ""),
        "warnings": warnings,
    }


def clean_voiceover_script(text: str) -> str:
    lines = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        cleaned = line.strip()
        if not cleaned:
            if lines and lines[-1] != "":
                lines.append("")
            continue
        if cleaned in {"---", "***"}:
            continue
        if cleaned.startswith("```"):
            continue
        # Remove common markdown styling while preserving the spoken sentence.
        if cleaned.startswith("**Transition:**"):
            cleaned = cleaned.replace("**Transition:**", "", 1).strip()
        elif cleaned.lower().startswith("transition:"):
            cleaned = cleaned.split(":", 1)[1].strip()
        cleaned = cleaned.replace("**", "").strip()
        if cleaned:
            lines.append(cleaned)
    return "\n".join(lines).strip()


def validate_voiceover_script_text(script: str) -> list[str]:
    warnings = []
    banned_prefixes = ("#", "##", "###", "target length:", "format:", "tone:")
    leaked_lines = [line for line in script.splitlines() if line.strip().lower().startswith(banned_prefixes)]
    if leaked_lines:
        warnings.append(f"Script may still contain headings/metadata lines: {leaked_lines[:5]}")
    if "[METADATA]" in script or "[NOTES]" in script or "CHATGPT_PIPELINE_RESPONSE_V1" in script:
        warnings.append("Script contains response wrapper markers")
    if "```" in script:
        warnings.append("Script contains markdown code fences")
    return warnings



## 1. Load App 4 Packet

This loads the existing `video_packaging_result.json`. Run this after setting `RESUME_RUN_ID` above.


In [4]:
video_packaging_result = load_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_result.json")
if video_packaging_result.get("status") != "ok":
    raise RuntimeError(video_packaging_result.get("summary", {}))
if not video_packaging_result.get("handoff", {}).get("ready_for_script_generation", False):
    raise RuntimeError("video_packaging_result is not ready for script generation")

video_packet = video_packaging_result["data"]["video_packet"]
print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {len(video_packet.get('stories', []))}")
print(f"Target seconds:         {video_packet.get('target_duration_seconds')}")


Video packaging status: ok
Selected format:        roundup
Selected stories:       7
Target seconds:         480


## 2. Create ChatGPT Content Packet Handoff

Paste the generated files into ChatGPT in this order: `01_instructions.md`, `02_payload.md`, then optionally `03_manifest_context.md`.


In [5]:
from src.pipeline_v2.handoffs.chatgpt_exporter import run as run_chatgpt_exporter

handoff_manifest = build_post_app4_manifest()
summary_bundle = build_summary_bundle()
chatgpt_output_dir = HANDOFF_OUTPUT_DIR / "chatgpt" / "content_packet"
chatgpt_handoff_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "profile": "content_packet",
        "prompt_version": "v1",
        "output_dir": str(chatgpt_output_dir),
    },
    "input": {
        "manifest": handoff_manifest,
        "artifacts": {
            "content_packet": video_packet,
        },
        "summaries": summary_bundle,
    },
}
chatgpt_handoff_result = run_chatgpt_exporter(chatgpt_handoff_input)
if chatgpt_handoff_result["status"] != "ok":
    raise RuntimeError(chatgpt_handoff_result["summary"])

chatgpt_paths = write_handoff_files(chatgpt_handoff_result["data"]["handoff_package"]["files"], chatgpt_output_dir)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_content_packet_handoff_input.json", chatgpt_handoff_input)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_content_packet_handoff_result.json", chatgpt_handoff_result)
write_json(HANDOFF_OUTPUT_DIR / "post_app4_manifest.json", handoff_manifest)
write_json(HANDOFF_OUTPUT_DIR / "post_app4_summary_bundle.json", summary_bundle)

print(f"ChatGPT handoff status: {chatgpt_handoff_result['status']}")
print(f"Files:                  {len(chatgpt_paths)}")
for path in chatgpt_paths:
    print(path)


ChatGPT handoff status: ok
Files:                  3
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/content_packet/01_instructions.md
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/content_packet/02_payload.md
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/content_packet/03_manifest_context.md


## 3. Manual ChatGPT Step

Paste the entire ChatGPT response into `CHATGPT_RESPONSE_PATH`. The response should use `CHATGPT_PIPELINE_RESPONSE_V1` with `[NOTES]`, `[SCRIPT]`, and `[METADATA]` sections. The notebook will extract only `[SCRIPT]` for voiceover.

Suggested reminder prompt after pasting the handoff:

```text
Return your answer in CHATGPT_PIPELINE_RESPONSE_V1 format with [NOTES], [SCRIPT], and [METADATA].
The [SCRIPT] block must contain only spoken narration text, no markdown headings, no timing labels, no visual notes, and no citations list.
```


In [6]:
CHATGPT_RESPONSE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not CHATGPT_RESPONSE_PATH.exists():
    CHATGPT_RESPONSE_PATH.write_text(PLACEHOLDER_RESPONSE_TEXT, encoding="utf-8")
    raise FileNotFoundError(f"Created placeholder response file. Fill it in, then rerun: {CHATGPT_RESPONSE_PATH}")
print(f"ChatGPT response file exists: {CHATGPT_RESPONSE_PATH}")


ChatGPT response file exists: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_scripts/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt_response.txt


## 4. Parse ChatGPT Response

This extracts the `[SCRIPT]` block, writes `chatgpt_full_script.txt`, and saves notes/metadata/warnings to `chatgpt_response_parsed.json`.


In [12]:
from src.pipeline_v2.voiceover.text_chunks import split_narration_into_chunks, word_count

chatgpt_response_text = CHATGPT_RESPONSE_PATH.read_text(encoding="utf-8")
parsed_chatgpt_response = parse_chatgpt_pipeline_response(chatgpt_response_text)
manual_script_text = parsed_chatgpt_response["script"]
manual_word_count = word_count(manual_script_text)

MANUAL_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
MANUAL_SCRIPT_PATH.write_text(manual_script_text + "\n", encoding="utf-8")
write_json(PARSED_RESPONSE_PATH, parsed_chatgpt_response)

print(f"Parsed script words: {manual_word_count}")
print(f"Parsed script path:  {MANUAL_SCRIPT_PATH}")
print(f"Parsed metadata:     {PARSED_RESPONSE_PATH}")
if parsed_chatgpt_response["warnings"]:
    print("Parser warnings:")
    for warning in parsed_chatgpt_response["warnings"]:
        print(f"- {warning}")


Parsed script words: 1224
Parsed script path:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_scripts/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt_full_script.txt
Parsed metadata:     /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_scripts/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt_response_parsed.json


## 5. Run Voiceover From Parsed Script

This uses `input.script_text` from the parsed `[SCRIPT]` block, so it does not require App 5 `script_generation_result`.

The cell reloads `src/pipeline_v2/config/workbench.yaml` every time it runs. That means you can change `voiceover.provider`, `voiceover.style`, `voiceover.voice`, `voiceover.speed`, or provider-specific settings in YAML, then rerun this cell to generate a new audio variant.


In [13]:
latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(latest_workbench_config, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])

voiceover_config = render_config_templates(deepcopy(latest_profile_config.get("voiceover", {})))
if not voiceover_config:
    raise KeyError("Workbench profile is missing voiceover config")
voiceover_output_dir = OUTPUT_V2 / "voiceover" / f"{RESUME_RUN_ID}_{voiceover_variant(voiceover_config)}_chatgpt_script"
voiceover_config["output_dir"] = str(voiceover_output_dir)

voiceover_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": voiceover_config,
    "input": {
        "script_text": manual_script_text,
        "script_source": str(MANUAL_SCRIPT_PATH),
    },
}

from src.pipeline_v2.voiceover.config import resolve_style
style_profile = resolve_style(voiceover_config["style"])
chunking = voiceover_config.get("chunking") or {}
resolved_chunking = {
    "max_chars": chunking.get("chunk_max_chars") or style_profile.chunk_max_chars,
    "max_sentences_per_chunk": chunking.get("max_sentences_per_chunk") or style_profile.max_sentences_per_chunk,
    "chunk_pause_seconds": (
        chunking.get("chunk_pause_seconds")
        if chunking.get("chunk_pause_seconds") is not None
        else style_profile.chunk_pause_seconds
    ),
    "paragraph_pause_seconds": (
        chunking.get("paragraph_pause_seconds")
        if chunking.get("paragraph_pause_seconds") is not None
        else style_profile.paragraph_pause_seconds
    ),
}
planned_chunks = split_narration_into_chunks(manual_script_text, **resolved_chunking)
if not planned_chunks:
    raise RuntimeError("Voiceover preflight produced zero chunks from manual script")

print(f"Voiceover config source: {PIPELINE_V2_CONFIG_PATH}")
print(f"Voiceover provider:      {voiceover_input['config']['provider']}")
print(f"Voiceover style:         {voiceover_input['config']['style']}")
print(f"Voiceover voice:         {voiceover_input['config'].get('voice') or '<style default>'}")
print(f"Voiceover speed:         {voiceover_input['config'].get('speed')}")
print(f"Voiceover output:        {voiceover_output_dir}")
print(f"Resolved chunking:       {resolved_chunking}")
print(f"Planned chunks:          {len(planned_chunks)}")

VOICEOVER_INPUT_DIR.mkdir(parents=True, exist_ok=True)
voiceover_input_path = VOICEOVER_INPUT_DIR / f"{RESUME_RUN_ID}_{voiceover_variant(voiceover_input['config'])}_chatgpt_script_voiceover_input.json"
write_json(voiceover_input_path, voiceover_input)

voiceover_python = PROJECT_ROOT / ".venv-voiceover" / "bin" / "python"
if not voiceover_python.exists():
    raise FileNotFoundError(f"Voiceover venv Python not found: {voiceover_python}")

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [
        str(voiceover_python),
        "-m",
        "src.pipeline_v2.voiceover.cli",
        "--input-json",
        str(voiceover_input_path),
    ],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

voiceover_summary = load_json(voiceover_output_dir / "summary.json")
voiceover_package = load_json(voiceover_output_dir / "voiceover_package.json")
voiceover_result = {
    "status": "ok",
    "data": {
        "run_id": RESUME_RUN_ID,
        "generated_at": voiceover_package.get("generated_at"),
        "voiceover_package": voiceover_package,
    },
    "summary": voiceover_summary,
    "handoff": voiceover_package.get("handoff", {}),
}
write_json(voiceover_output_dir / "voiceover_input.json", voiceover_input)
write_json(voiceover_output_dir / "voiceover_result.json", voiceover_result)

print(f"Voiceover status: {voiceover_summary['status']}")
print(f"Voice:            {voiceover_summary['voice']}")
print(f"Duration seconds: {voiceover_summary['duration_seconds']}")
print(f"Chunks:           {voiceover_summary['chunk_count']}")
print(f"Audio path:       {voiceover_summary['audio_path']}")
print(f"Wrote:            {voiceover_output_dir}")


Voiceover config source: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/src/pipeline_v2/config/workbench.yaml
Voiceover provider:      chatterbox_local
Voiceover style:         dramatic_news
Voiceover voice:         <style default>
Voiceover speed:         0.96
Voiceover output:        /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script
Resolved chunking:       {'max_chars': 420, 'max_sentences_per_chunk': 1, 'chunk_pause_seconds': 0.22, 'paragraph_pause_seconds': 0.65}
Planned chunks:          65


/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/.venv-voiceover/lib/python3.12/site-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/.venv-voiceover/lib/python3.12/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000


/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
`sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.
Sampling:   4%|▎         | 37/1000 [00:05<02:21,  6.82it/s]
/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
Sampling:   7%|▋         | 69/1000 [00:04<00:56, 16.37it/s]


{
  "provider": "chatterbox_local",
  "voice": "chatterbox_default",
  "style": "dramatic_news",
  "audio_path": "output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script/audio/full_voiceover.wav",
  "duration_seconds": 365.32,
  "chunk_count": 65,
  "word_count": 1224,
  "run_id": "v2_apps_2026_06_26_231604_daily_48h_all_sources",
  "status": "ready",
  "ready_for_video_assembly": true,
  "requires_manual_audio_review": true
}
Voiceover status: ready
Voice:            chatterbox_default
Duration seconds: 365.32
Chunks:           65
Audio path:       output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script/audio/full_voiceover.wav
Wrote:            /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice

In [14]:
voiceover_summary

{'run_id': 'v2_apps_2026_06_26_231604_daily_48h_all_sources',
 'status': 'ready',
 'provider': 'chatterbox_local',
 'voice': 'chatterbox_default',
 'style': 'dramatic_news',
 'audio_path': 'output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script/audio/full_voiceover.wav',
 'duration_seconds': 365.32,
 'chunk_count': 65,
 'word_count': 1224,
 'ready_for_video_assembly': True,
 'requires_manual_audio_review': True}

## 6. Plan Motion Visual Scenes

This planner creates a motion-oriented scene plan from the parsed script, selected stories, and voiceover timing. It supports screenshots, logos, kinetic text, bullet builds, quote highlights, comparisons, recaps, data counters, trend lines, ecosystem maps, architecture flows, and transition stingers.

If you want ChatGPT to manually plan visuals instead, run section `6A`, paste the response, then run section `6B`. Section `7` works with either path because both produce the same `visual_scene_planning_result` envelope.


In [15]:
from src.pipeline_v2.visuals.planner import run as run_visual_scene_planner

latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(latest_workbench_config, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])

visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_planner_config = render_config_templates(deepcopy(visuals_config.get("planner", {})))
if not visual_planner_config:
    raise KeyError("Workbench profile is missing visuals.planner config")

if "manual_script_text" not in globals() or not manual_script_text.strip():
    if not MANUAL_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Manual script file not found: {MANUAL_SCRIPT_PATH}")
    manual_script_text = MANUAL_SCRIPT_PATH.read_text(encoding="utf-8").strip()

if "video_packaging_result" not in globals():
    video_packaging_result = load_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_result.json")

if "voiceover_package" not in globals():
    voiceover_candidates = sorted(
        (OUTPUT_V2 / "voiceover").glob(f"{RESUME_RUN_ID}_*chatgpt_script/voiceover_package.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not voiceover_candidates:
        raise FileNotFoundError(f"No voiceover_package.json found for {RESUME_RUN_ID}")
    voiceover_package = load_json(voiceover_candidates[0])
    print(f"Loaded latest voiceover package: {voiceover_candidates[0]}")

visual_scene_planning_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_planner_config,
    "input": {
        "video_packaging": video_packaging_result,
        "script_text": manual_script_text,
        "script_source": str(MANUAL_SCRIPT_PATH),
        "voiceover": voiceover_package,
    },
}

visual_scene_planning_result = run_visual_scene_planner(visual_scene_planning_input)
visual_scene_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "planning"
visual_scene_output_dir.mkdir(parents=True, exist_ok=True)

visual_scene_plan = visual_scene_planning_result["data"]["visual_scene_plan"]
write_json(visual_scene_output_dir / "visual_scene_planning_input.json", visual_scene_planning_input)
write_json(visual_scene_output_dir / "visual_scene_planning_result.json", visual_scene_planning_result)
write_json(visual_scene_output_dir / "visual_scene_plan.json", visual_scene_plan)
write_json(visual_scene_output_dir / "scene_timeline.json", {"scene_timeline": visual_scene_plan.get("scene_timeline", [])})
write_json(visual_scene_output_dir / "asset_manifest.json", visual_scene_plan.get("asset_manifest", {}))
write_json(visual_scene_output_dir / "story_sections.json", {"story_sections": visual_scene_plan.get("story_sections", [])})
write_json(visual_scene_output_dir / "coverage.json", visual_scene_plan.get("coverage", {}))
write_json(visual_scene_output_dir / "summary.json", visual_scene_planning_result.get("summary", {}))

llm_calls_path = visual_scene_output_dir / "llm_calls.jsonl"
llm_calls_path.write_text(
    "".join(json.dumps(call, ensure_ascii=False) + "\n" for call in visual_scene_planning_result.get("llm_calls", [])),
    encoding="utf-8",
)

print(f"Visual scene planning status: {visual_scene_planning_result['status']}")
print(f"Scenes:                       {visual_scene_planning_result['summary']['scene_count']}")
print(f"Story sections:               {visual_scene_planning_result['summary'].get('story_section_count')}")
print(f"Source article scenes:        {visual_scene_planning_result['summary']['source_article_scene_count']}")
print(f"Logo reveal scenes:           {visual_scene_planning_result['summary']['logo_reveal_scene_count']}")
print(f"Kinetic type scenes:          {visual_scene_planning_result['summary']['kinetic_type_scene_count']}")
coverage = visual_scene_planning_result['summary'].get('coverage', {})
print(f"Covered chunks:               {coverage.get('covered_unit_count')}/{coverage.get('expected_unit_count')}")
print(f"Max scene seconds:            {coverage.get('max_scene_duration_seconds')}")
print(f"LLM calls:                    {visual_scene_planning_result['summary']['llm_call_count']}")
print(f"Output dir:                   {visual_scene_output_dir}")
if visual_scene_planning_result["status"] != "ok":
    raise RuntimeError(visual_scene_planning_result["summary"].get("failure_reason", "visual scene planning failed"))


/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Visual scene planning status: ok
Scenes:                       53
Story sections:               7
Source article scenes:        2
Logo reveal scenes:           3
Kinetic type scenes:          15
Covered chunks:               65/65
Max scene seconds:            14.0
LLM calls:                    8
Output dir:                   /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/planning


In [16]:
visual_scene_planning_result["summary"]


{'script_word_count': 1224,
 'voiceover_duration_seconds': 365.32,
 'story_count': 7,
 'scene_count': 53,
 'source_article_scene_count': 2,
 'logo_reveal_scene_count': 3,
 'kinetic_type_scene_count': 15,
 'llm_call_count': 8,
 'failure_reason': '',
 'story_section_count': 7,
 'source_backed_scene_count': 5,
 'bullet_build_scene_count': 4,
 'quote_or_claim_highlight_scene_count': 3,
 'entity_compare_scene_count': 2,
 'recap_sequence_scene_count': 4,
 'data_counter_scene_count': 9,
 'trend_line_scene_count': 3,
 'bar_chart_scene_count': 0,
 'bar_race_scene_count': 0,
 'ranked_list_scene_count': 0,
 'benchmark_delta_scene_count': 3,
 'ecosystem_map_scene_count': 0,
 'architecture_flow_scene_count': 2,
 'process_zoom_scene_count': 1,
 'browser_walkthrough_scene_count': 0,
 'decision_tree_scene_count': 0,
 'capability_matrix_scene_count': 0,
 'timeline_zoom_scene_count': 0,
 'document_stack_scene_count': 0,
 'signal_radar_scene_count': 0,
 'interface_transform_scene_count': 2,
 'transition_

## 6A. Optional: Export ChatGPT Visual Planning Handoff

Run this when you want ChatGPT to create the visual scene plan manually. Paste the generated files into ChatGPT, save the response to `chatgpt_visual_plan_response.txt`, then run section `6B`.


In [7]:
from src.pipeline_v2.handoffs.chatgpt_exporter import run as run_chatgpt_exporter

latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])
visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_planner_config = render_config_templates(deepcopy(visuals_config.get("planner", {})))

if "manual_script_text" not in globals() or not manual_script_text.strip():
    if not MANUAL_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Manual script file not found: {MANUAL_SCRIPT_PATH}")
    manual_script_text = MANUAL_SCRIPT_PATH.read_text(encoding="utf-8").strip()

if "video_packaging_result" not in globals():
    video_packaging_result = load_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_result.json")

if "voiceover_package" not in globals():
    voiceover_candidates = sorted(
        (OUTPUT_V2 / "voiceover").glob(f"{RESUME_RUN_ID}_*chatgpt_script/voiceover_package.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not voiceover_candidates:
        raise FileNotFoundError(f"No voiceover_package.json found for {RESUME_RUN_ID}")
    voiceover_package = load_json(voiceover_candidates[0])
    print(f"Loaded latest voiceover package: {voiceover_candidates[0]}")


def compact_voiceover_timing(package: dict) -> dict:
    pkg = package.get("data", {}).get("voiceover_package") if isinstance(package.get("data"), dict) else None
    if not isinstance(pkg, dict):
        pkg = package.get("voiceover_package") if isinstance(package.get("voiceover_package"), dict) else package
    audio = pkg.get("audio", {}) if isinstance(pkg.get("audio"), dict) else {}
    chunks = pkg.get("chunks", []) if isinstance(pkg.get("chunks"), list) else []
    return {
        "duration_seconds": audio.get("duration_seconds") or pkg.get("duration_seconds"),
        "audio_path": audio.get("full_audio_path") or audio.get("audio_path") or pkg.get("audio_path"),
        "chunk_count": len(chunks),
        "chunks": [
            {
                "chunk_id": chunk.get("chunk_id"),
                "order": chunk.get("order"),
                "start_seconds": chunk.get("start_seconds"),
                "end_seconds": chunk.get("end_seconds"),
                "duration_seconds": chunk.get("duration_seconds"),
                "text": chunk.get("text"),
            }
            for chunk in chunks
            if isinstance(chunk, dict)
        ],
    }

video_packet = video_packaging_result["data"]["video_packet"]
visual_handoff_packet = {
    "package_type": "chatgpt_visual_scene_plan_packet",
    "selected_format": video_packaging_result["data"].get("selected_format", video_packet.get("selected_format", "roundup")),
    "run_id": RESUME_RUN_ID,
    "script": {
        "script_source": str(MANUAL_SCRIPT_PATH),
        "word_count": len(manual_script_text.split()),
        "full_text": manual_script_text,
    },
    "voiceover_timing": compact_voiceover_timing(voiceover_package),
    "video_packet": video_packet,
    "visual_context": {
        "scene_policy": visual_planner_config.get("scene_policy", {}),
        "visual_capabilities": visual_planner_config.get("visual_capabilities", {}),
        "style": visual_planner_config.get("style", {}),
    },
    "output_contract": {
        "response_marker": "CHATGPT_VISUAL_PLAN_RESPONSE_V1",
        "json_section": "[VISUAL_SCENE_PLAN_JSON]",
        "required_scene_fields": [
            "scene_id", "scene_order", "story_id", "section_type", "start_seconds", "end_seconds", "render_kind", "script_excerpt", "visual_intent", "payload"
        ],
        "source_backed_scene_types": ["source_article_scene", "quote_or_claim_highlight_scene"],
        "old_card_types_rejected": ["title_card", "timeline_card", "entity_card", "recap_card", "fallback_card", "quote_card", "data_card"],
    },
}

visual_chatgpt_output_dir = HANDOFF_OUTPUT_DIR / "chatgpt" / "visual_scene_plan"
visual_chatgpt_response_path = OUTPUT_V2 / "manual_visual_plans" / RESUME_RUN_ID / "chatgpt_visual_plan_response.txt"
visual_chatgpt_response_path.parent.mkdir(parents=True, exist_ok=True)
if not visual_chatgpt_response_path.exists():
    visual_chatgpt_response_path.write_text("Paste the full CHATGPT_VISUAL_PLAN_RESPONSE_V1 response here, then run section 6B.\n", encoding="utf-8")

visual_handoff_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "profile": "visual_scene_plan",
        "prompt_version": "v1",
        "output_dir": str(visual_chatgpt_output_dir),
    },
    "input": {
        "manifest": build_post_app4_manifest(),
        "artifacts": {"visual_scene_plan": visual_handoff_packet},
        "summaries": build_summary_bundle(),
    },
}

visual_handoff_result = run_chatgpt_exporter(visual_handoff_input)
if visual_handoff_result["status"] != "ok":
    raise RuntimeError(visual_handoff_result["summary"])
visual_handoff_paths = write_handoff_files(visual_handoff_result["data"]["handoff_package"]["files"], visual_chatgpt_output_dir)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_visual_scene_plan_handoff_input.json", visual_handoff_input)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_visual_scene_plan_handoff_result.json", visual_handoff_result)
write_json(visual_chatgpt_output_dir / "visual_handoff_packet.json", visual_handoff_packet)

print(f"Visual ChatGPT handoff status: {visual_handoff_result['status']}")
print(f"Files:                         {len(visual_handoff_paths)}")
for path in visual_handoff_paths:
    print(path)
print(f"Response paste file:           {visual_chatgpt_response_path}")


Loaded latest voiceover package: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script/voiceover_package.json
Visual ChatGPT handoff status: ok
Files:                         3
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/visual_scene_plan/01_instructions.md
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/visual_scene_plan/02_payload.md
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/handoffs/v2_apps_2026_06_26_231604_daily_48h_all_sources/chatgpt/visual_scene_plan/03_manifest_context.md
Response paste file:           /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/manual_visual_plans/v2_apps

## 6B. Optional: Parse ChatGPT Visual Plan Response

Run this after saving ChatGPT’s `CHATGPT_VISUAL_PLAN_RESPONSE_V1` response. It writes the parsed plan into the normal visual planning output folder, so section `7` can run unchanged.


In [8]:
from src.pipeline_v2.handoffs.chatgpt_visual_plan_parser import run as run_chatgpt_visual_plan_parser

latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])
visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_planner_config = render_config_templates(deepcopy(visuals_config.get("planner", {})))

if "manual_script_text" not in globals() or not manual_script_text.strip():
    if not MANUAL_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Manual script file not found: {MANUAL_SCRIPT_PATH}")
    manual_script_text = MANUAL_SCRIPT_PATH.read_text(encoding="utf-8").strip()

if "video_packaging_result" not in globals():
    video_packaging_result = load_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_result.json")

if "voiceover_package" not in globals():
    voiceover_candidates = sorted(
        (OUTPUT_V2 / "voiceover").glob(f"{RESUME_RUN_ID}_*chatgpt_script/voiceover_package.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not voiceover_candidates:
        raise FileNotFoundError(f"No voiceover_package.json found for {RESUME_RUN_ID}")
    voiceover_package = load_json(voiceover_candidates[0])

visual_chatgpt_response_path = OUTPUT_V2 / "manual_visual_plans" / RESUME_RUN_ID / "chatgpt_visual_plan_response.txt"
if not visual_chatgpt_response_path.exists():
    visual_chatgpt_response_path.parent.mkdir(parents=True, exist_ok=True)
    visual_chatgpt_response_path.write_text("Paste the full CHATGPT_VISUAL_PLAN_RESPONSE_V1 response here, then rerun this cell.\n", encoding="utf-8")
    raise FileNotFoundError(f"Created placeholder visual plan response file. Fill it in, then rerun: {visual_chatgpt_response_path}")
response_text = visual_chatgpt_response_path.read_text(encoding="utf-8")
if "CHATGPT_VISUAL_PLAN_RESPONSE_V1" not in response_text:
    raise ValueError(f"Visual plan response file does not contain CHATGPT_VISUAL_PLAN_RESPONSE_V1: {visual_chatgpt_response_path}")

visual_scene_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "planning"
visual_scene_output_dir.mkdir(parents=True, exist_ok=True)
visual_parse_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "scene_policy": visual_planner_config.get("scene_policy", {}),
        "visual_capabilities": visual_planner_config.get("visual_capabilities", {}),
        "require_complete_timeline": True,
        "gap_tolerance_seconds": 0.5,
        "output_dir": str(visual_scene_output_dir),
    },
    "input": {
        "response_text": response_text,
        "video_packaging": video_packaging_result,
        "script_text": manual_script_text,
        "script_source": str(MANUAL_SCRIPT_PATH),
        "voiceover": voiceover_package,
    },
}

visual_scene_planning_result = run_chatgpt_visual_plan_parser(visual_parse_input)
visual_scene_plan = visual_scene_planning_result["data"]["visual_scene_plan"]
write_json(visual_scene_output_dir / "chatgpt_visual_plan_parse_input.json", visual_parse_input)
write_json(visual_scene_output_dir / "visual_scene_planning_result.json", visual_scene_planning_result)
write_json(visual_scene_output_dir / "visual_scene_plan.json", visual_scene_plan)
write_json(visual_scene_output_dir / "scene_timeline.json", {"scene_timeline": visual_scene_plan.get("scene_timeline", [])})
write_json(visual_scene_output_dir / "asset_manifest.json", visual_scene_plan.get("asset_manifest", {}))
write_json(visual_scene_output_dir / "story_sections.json", {"story_sections": visual_scene_plan.get("story_sections", [])})
write_json(visual_scene_output_dir / "coverage.json", visual_scene_plan.get("coverage", {}))
write_json(visual_scene_output_dir / "summary.json", visual_scene_planning_result.get("summary", {}))

print(f"Parsed visual plan status: {visual_scene_planning_result['status']}")
print(f"Scenes:                    {visual_scene_planning_result['summary']['scene_count']}")
print(f"Source-backed scenes:      {visual_scene_planning_result['summary']['source_backed_scene_count']}")
print(f"Transition stingers:       {visual_scene_planning_result['summary'].get('transition_stinger_scene_count', 0)}")
print(f"Output dir:                {visual_scene_output_dir}")
if visual_scene_planning_result["status"] != "ok":
    raise RuntimeError(visual_scene_planning_result["summary"].get("failure_reason", "visual plan parse failed"))


Parsed visual plan status: ok
Scenes:                    61
Source-backed scenes:      11
Transition stingers:       1
Output dir:                /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/planning


## 7. Discover Media + Resolve Visual Assets

This step searches approved media providers for contextual images and b-roll, records attribution metadata, then captures source screenshots and resolves logo/media assets. Code-generated scenes still need no external files.


In [ ]:
latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(latest_workbench_config, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])

visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_media_discovery_config = render_config_templates(deepcopy(visuals_config.get("media_discovery", {})))
visual_asset_resolver_config = render_config_templates(deepcopy(visuals_config.get("asset_resolver", {})))
if not visual_media_discovery_config:
    raise KeyError("Workbench profile is missing visuals.media_discovery config")
if not visual_asset_resolver_config:
    raise KeyError("Workbench profile is missing visuals.asset_resolver config")

visual_media_discovery_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "media_discovery"
visual_asset_resolution_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "asset_resolution"
visual_media_discovery_config["output_dir"] = str(visual_media_discovery_output_dir)
visual_asset_resolver_config["output_dir"] = str(visual_asset_resolution_output_dir)
if not visual_asset_resolver_config.get("logos", {}).get("local_logo_dir"):
    visual_asset_resolver_config.setdefault("logos", {})["local_logo_dir"] = str(PROJECT_ROOT / "src" / "pipeline_v2" / "visuals" / "remotion_engine" / "assets" / "logos")

if "visual_scene_planning_result" not in globals():
    visual_scene_planning_result = load_json(OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "planning" / "visual_scene_planning_result.json")
if visual_scene_planning_result.get("status") != "ok":
    raise RuntimeError(visual_scene_planning_result.get("summary", {}))

visual_input_dir = OUTPUT_V2 / "visual_inputs"
visual_media_discovery_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_media_discovery_config,
    "input": {"visual_scene_planning": visual_scene_planning_result},
}
visual_media_discovery_input_path = visual_input_dir / f"{RESUME_RUN_ID}_visual_media_discovery_input.json"
write_json(visual_media_discovery_input_path, visual_media_discovery_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [sys.executable, "-m", "src.pipeline_v2.visuals.media_discovery.cli", "--input-json", str(visual_media_discovery_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)
visual_media_discovery_result = load_json(visual_media_discovery_output_dir / "visual_media_discovery_result.json")
visual_media_discovery_summary = load_json(visual_media_discovery_output_dir / "summary.json")
if not visual_media_discovery_result.get("handoff", {}).get("ready_for_visual_asset_resolution", False):
    raise RuntimeError(visual_media_discovery_summary)

visual_asset_resolution_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_asset_resolver_config,
    "input": {
        "visual_scene_planning": visual_scene_planning_result,
        "visual_media_discovery": visual_media_discovery_result,
    },
}

visual_asset_resolution_input_path = visual_input_dir / f"{RESUME_RUN_ID}_visual_asset_resolution_input.json"
write_json(visual_asset_resolution_input_path, visual_asset_resolution_input)

subprocess.run(
    [sys.executable, "-m", "src.pipeline_v2.visuals.asset_resolver.cli", "--input-json", str(visual_asset_resolution_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

visual_asset_resolution_result = load_json(visual_asset_resolution_output_dir / "visual_asset_resolution_result.json")
visual_asset_manifest = load_json(visual_asset_resolution_output_dir / "visual_asset_manifest.json")
visual_asset_resolution_summary = load_json(visual_asset_resolution_output_dir / "summary.json")

print(f"Media discovery status: {visual_media_discovery_result['status']}")
print(f"Selected images:        {visual_media_discovery_summary['selected_image_count']}")
print(f"Selected videos:        {visual_media_discovery_summary['selected_video_count']}")
print(f"Explicit fallbacks:     {visual_media_discovery_summary['fallback_scene_count']}")
print(f"Download errors:        {visual_media_discovery_summary.get('download_error_count', 0)}")
print(f"Visual asset status:    {visual_asset_resolution_result['status']}")
print(f"Resolved assets:        {visual_asset_resolution_summary['resolved_asset_count']}")
print(f"Source screenshots:     {visual_asset_resolution_summary['source_screenshot_count']}")
print(f"Logo assets:            {visual_asset_resolution_summary['logo_asset_count']}")
print(f"Context images:         {visual_asset_resolution_summary.get('context_image_count', 0)}")
print(f"B-roll videos:          {visual_asset_resolution_summary.get('broll_video_count', 0)}")
print(f"Failed assets:          {visual_asset_resolution_summary['failed_asset_count']}")
print(f"Output dir:             {visual_asset_resolution_output_dir}")
if not visual_asset_resolution_result.get("handoff", {}).get("ready_for_visual_render", False):
    raise RuntimeError(visual_asset_resolution_summary)


In [5]:
visual_asset_resolution_summary


{'requested_scene_count': 61,
 'resolved_asset_count': 61,
 'source_screenshot_count': 11,
 'logo_asset_count': 7,
 'failed_asset_count': 0,
 'asset_type_counts': {'none': 43, 'logo_asset': 7, 'source_screenshot': 11},
 'status_counts': {'ready': 43, 'resolved': 18},
 'output_dir': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/asset_resolution',
 'failure_reason': ''}

## 8. Render Visual Scene Clips

This step renders one MP4 per visual scene using the local Remotion engine. The next step assembles those scene clips with the full voiceover.


In [6]:
latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(latest_workbench_config, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])

visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_render_config = render_config_templates(deepcopy(visuals_config.get("render", {})))
if not visual_render_config:
    raise KeyError("Workbench profile is missing visuals.render config")

visual_render_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "render"
visual_render_config["output_dir"] = str(visual_render_output_dir)
if not visual_render_config.get("remotion_engine_dir"):
    visual_render_config["remotion_engine_dir"] = str(PROJECT_ROOT / "src" / "pipeline_v2" / "visuals" / "remotion_engine")

if "visual_asset_resolution_result" not in globals():
    visual_asset_resolution_result = load_json(OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "asset_resolution" / "visual_asset_resolution_result.json")
if visual_asset_resolution_result.get("status") not in {"ok", "partial"}:
    raise RuntimeError(visual_asset_resolution_result.get("summary", {}))

visual_render_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_render_config,
    "input": {"visual_asset_resolution": visual_asset_resolution_result},
}

visual_render_input_dir = OUTPUT_V2 / "visual_inputs"
visual_render_input_path = visual_render_input_dir / f"{RESUME_RUN_ID}_visual_render_input.json"
write_json(visual_render_input_path, visual_render_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [sys.executable, "-m", "src.pipeline_v2.visuals.render.cli", "--input-json", str(visual_render_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

visual_render_result = load_json(visual_render_output_dir / "visual_render_result.json")
visual_render_package = load_json(visual_render_output_dir / "visual_render_package.json")
visual_render_summary = load_json(visual_render_output_dir / "render_summary.json")

print(f"Visual render status: {visual_render_result['status']}")
print(f"Rendered scenes:      {visual_render_summary['rendered_scene_count']}")
print(f"Failed scenes:        {visual_render_summary['failed_scene_count']}")
print(f"Output dir:           {visual_render_output_dir}")
if not visual_render_result.get("handoff", {}).get("ready_for_video_assembly", False):
    raise RuntimeError(visual_render_summary)


Visual render status: ok
Rendered scenes:      61
Failed scenes:        0
Output dir:           /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/render


In [7]:
visual_render_summary

{'requested_scene_count': 61,
 'rendered_scene_count': 61,
 'failed_scene_count': 0,
 'warning_count': 0,
 'render_kind_counts': {'kinetic_type_scene': 6,
  'entity_compare_scene': 1,
  'transition_stinger_scene': 1,
  'logo_reveal_scene': 7,
  'source_article_scene': 7,
  'interface_transform_scene': 5,
  'architecture_flow_scene': 5,
  'bullet_build_scene': 9,
  'quote_or_claim_highlight_scene': 4,
  'ecosystem_map_scene': 3,
  'data_counter_scene': 1,
  'signal_radar_scene': 3,
  'decision_tree_scene': 5,
  'benchmark_delta_scene': 2,
  'recap_sequence_scene': 2},
 'status_counts': {'rendered': 61},
 'output_dir': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/render',
 'render_settings': {'quality_preset': '2k',
  'width': 2560,
  'height': 1440,
  'fps': 30,
  'crf': 15,
  'video_bitrate': None,
  'x264_preset': 'medium',
  'pixel_format': 'yuv420p',
  'worker_count': 3},
 'failure_reason': ''}

## 9. Assemble Full Video

This step stitches the rendered scene clips in order and muxes them with the full voiceover WAV into one reviewable MP4.


In [8]:
latest_workbench_config = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(latest_workbench_config, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
latest_run_profile = latest_workbench_config.get("default_run_profile", RUN_PROFILE)
latest_profile_config = deepcopy(latest_workbench_config["profiles"][latest_run_profile])

visuals_config = deepcopy(latest_profile_config.get("visuals", {}))
visual_assembly_config = render_config_templates(deepcopy(visuals_config.get("assembly", {})))
if not visual_assembly_config:
    raise KeyError("Workbench profile is missing visuals.assembly config")

visual_assembly_output_dir = OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "assembly"
visual_assembly_config["output_dir"] = str(visual_assembly_output_dir)

if "visual_render_result" not in globals():
    visual_render_result = load_json(OUTPUT_V2 / "visuals" / RESUME_RUN_ID / "render" / "visual_render_result.json")

if "voiceover_package" not in globals():
    voiceover_candidates = sorted(
        (OUTPUT_V2 / "voiceover").glob(f"{RESUME_RUN_ID}*/voiceover_package.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not voiceover_candidates:
        raise FileNotFoundError(f"No voiceover_package.json found for run prefix: {RESUME_RUN_ID}")
    voiceover_package = load_json(voiceover_candidates[0])
    print(f"Loaded voiceover package: {voiceover_candidates[0]}")

visual_assembly_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_assembly_config,
    "input": {
        "visual_render": visual_render_result,
        "voiceover": voiceover_package,
    },
}

visual_assembly_input_dir = OUTPUT_V2 / "visual_inputs"
visual_assembly_input_path = visual_assembly_input_dir / f"{RESUME_RUN_ID}_visual_assembly_input.json"
write_json(visual_assembly_input_path, visual_assembly_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [sys.executable, "-m", "src.pipeline_v2.visuals.assembly.cli", "--input-json", str(visual_assembly_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

video_assembly_result = load_json(visual_assembly_output_dir / "video_assembly_result.json")
final_video_package = load_json(visual_assembly_output_dir / "final_video_package.json")
video_assembly_summary = load_json(visual_assembly_output_dir / "assembly_summary.json")

print(f"Video assembly status: {video_assembly_result['status']}")
print(f"Scene clips:           {video_assembly_summary['assembled_scene_count']}")
print(f"Audio duration:        {video_assembly_summary['audio_duration_seconds']}s")
print(f"Final duration:        {video_assembly_summary['final_duration_seconds']}s")
print(f"Final video:           {video_assembly_summary['final_video_path']}")
if not video_assembly_result.get("handoff", {}).get("ready_for_manual_review", False):
    raise RuntimeError(video_assembly_summary)


Loaded voiceover package: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/voiceover/v2_apps_2026_06_26_231604_daily_48h_all_sources_chatterbox_local_dramatic_news_default_voice_speed_0p96_chatgpt_script/voiceover_package.json
Video assembly status: ok
Scene clips:           61
Audio duration:        365.32s
Final duration:        365.32s
Final video:           /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/assembly/final_video.mp4


In [9]:
video_assembly_summary


{'assembled_scene_count': 61,
 'audio_duration_seconds': 365.32,
 'visual_duration_seconds': 365.32,
 'final_duration_seconds': 365.32,
 'duration_delta_seconds': 0.0,
 'output_dir': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/assembly',
 'final_video_path': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visuals/v2_apps_2026_06_26_231604_daily_48h_all_sources/assembly/final_video.mp4',
 'failure_reason': ''}